In [34]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as snb


In [53]:
df_orders = pd.read_excel('./data/Muesli Project raw data.xlsx', sheet_name='Orders', header=1)
df_campaign = pd.read_excel('./data/Muesli Project raw data.xlsx', sheet_name='Campaign Data')
df_process = pd.read_excel('./data/Muesli Project raw data.xlsx', sheet_name='Order Process Data')
df_intern = pd.read_excel('./data/Muesli Project raw data.xlsx', sheet_name='InternData Study')

In [ ]:
# Isolate all duplicate rows (showing all copies)
duplicates_df = df_intern[df_intern.duplicated(keep=False)]

# Sort by Order ID so the identical rows sit right next to each other
duplicates_df_sorted = duplicates_df.sort_values(by='Order ID')

# View the first 10 rows to inspect them
duplicates_df_sorted.head(10)


,Order ID,Ready to Ship Date,Pickup Date
78,CA-2019-100244,2019-09-24,2019-09-25
79,CA-2019-100244,2019-09-24,2019-09-25
80,CA-2019-100244,2019-09-24,2019-09-25
82,CA-2019-100244,2019-09-24,2019-09-25
83,CA-2019-100244,2019-09-24,2019-09-25
59,CA-2019-102162,2019-09-16,2019-09-18
61,CA-2019-102162,2019-09-16,2019-09-18
4,CA-2019-106278,2019-09-05,2019-09-06
6,CA-2019-106278,2019-09-05,2019-09-06
10,CA-2019-106950,2019-09-06,2019-09-09


In [56]:
# Check how many rows are exact duplicates across all columns
print("Duplicate rows count:", df_orders.duplicated().sum())
print("Duplicate rows count:", df_campaign.duplicated().sum())
print("Duplicate rows count:", df_process.duplicated(subset=['Order ID']).sum())
print("Duplicate rows count:", df_intern.duplicated().sum())

# Drop the duplicate rows if found, keeping only the first one
df_orders = df_orders.drop_duplicates()
df_campaign = df_campaign.drop_duplicates()
df_process = df_process.drop(columns=['Row ID']).drop_duplicates(subset=['Order ID'], keep='first')
df_intern = df_intern.drop_duplicates()

Duplicate rows count: 0
Duplicate rows count: 0
Duplicate rows count: 2897
Duplicate rows count: 86


In [60]:
# Get a clear list of every column and how many missing values it has
print(df_orders.isnull().sum())
#print(df_campaign.isnull().sum())
#print(df_process.isnull().sum())
#print(df_intern.isnull().sum())



Index              0
Order ID           0
Order Date         0
Ship Mode          0
Customer ID        0
Customer Name      0
Origin Channel     0
Country/Region     0
City               0
State              0
Postal Code       11
Region             0
Category           0
Sub-Category       0
Product ID         0
Sales              0
Quantity           0
Discount           0
Profit             0
dtype: int64


In [65]:
# Force the entire Postal Code column to be text (strings) first
df_orders['Postal Code'] = df_orders['Postal Code'].astype(str)
# Directly assign the correct postal code to Burlington, Vermont rows
df_orders.loc[(df_orders['City'] == 'Burlington') & (df_orders['State'] == 'Vermont'), 'Postal Code'] = '05401'

# Make sure the rest of your postal codes are clean 5-digit strings (removes any '.0')
df_orders['Postal Code'] = df_orders['Postal Code'].astype(str).str.split('.').str[0].str.zfill(5)

# View all rows where the City is Burlington to check the new Postal Codes
#df_orders[df_orders['City'] == 'Burlington'][['City', 'State', 'Postal Code']].head(10)

In [19]:
# 1. Start by copying your main orders dataframe
df_master = df_orders.copy()

# 2. Merge the process sheet (Truck scans)
# We pull 'Order ID' to match on, and 'On Truck Scan Date' to add
df_master = pd.merge(df_master, df_process[['Order ID', 'On Truck Scan Date']], on='Order ID', how='left')

# 3. Merge the intern sheet (Ready to ship scans)
df_master = pd.merge(df_master, df_intern[['Order ID', 'Ready to Ship Date']], on='Order ID', how='left')

# 4. Merge the campaign sheet (Arrival scans)
df_master = pd.merge(df_master, df_campaign[['Order ID', 'Arrival Scan Date']], on='Order ID', how='left')

# 5. Check how many records now have data across the timeline
print("=== Master Dataframe Completion Check ===")
print(df_master[['Order Date', 'Ready to Ship Date', 'On Truck Scan Date', 'Arrival Scan Date']].notnull().sum())


=== Master Dataframe Completion Check ===
Order Date            23171
Ready to Ship Date     2771
On Truck Scan Date    19076
Arrival Scan Date      2211
dtype: int64


In [68]:
df_master.head(2)

,Index,Order ID,Order Date,Ship Mode,Customer ID,Customer Name,Origin Channel,Country/Region,City,State,...,Profit,On Truck Scan Date,Ready to Ship Date,Arrival Scan Date,Days_Order_to_Ready,Days_Ready_to_Truck,Total_Warehouse_Days,Transit_Days,Order_Day_of_Week,Order_Day_Name
0,27,CA-2019-121755,2019-01-16,Second Class,EH-13945,Eric Hoffmann,Email,United States,Los Angeles,California,...,11.7741,2019-01-23,NaT,NaT,NaN,NaN,7.0,NaN,2,Wednesday
1,27,CA-2019-121755,2019-01-16,Second Class,EH-13945,Eric Hoffmann,Email,United States,Los Angeles,California,...,11.7741,2019-01-23,NaT,NaT,NaN,NaN,7.0,NaN,2,Wednesday


In [20]:
# 1. Calculate the exact days spent in each part of the supply chain
df_master['Days_Order_to_Ready'] = (df_master['Ready to Ship Date'] - df_master['Order Date']).dt.days
df_master['Days_Ready_to_Truck'] = (df_master['On Truck Scan Date'] - df_master['Ready to Ship Date']).dt.days
df_master['Total_Warehouse_Days'] = (df_master['On Truck Scan Date'] - df_master['Order Date']).dt.days

# 2. Calculate the customer transit time (where available)
df_master['Transit_Days'] = (df_master['Arrival Scan Date'] - df_master['On Truck Scan Date']).dt.days

# 3. Print the baseline business realities
print("=== BASELINE LOGISTICS PERFORMANCE ===")
print(f"Average Warehouse Processing Time: {df_master['Days_Order_to_Ready'].mean():.2f} days")
print(f"Average Wait Time for a Truck:     {df_master['Days_Ready_to_Truck'].mean():.2f} days")
print(f"Average Total Time in Warehouse:   {df_master['Total_Warehouse_Days'].mean():.2f} days")
print(f"Average Shipping Transit Time:     {df_master['Transit_Days'].mean():.2f} days")


=== BASELINE LOGISTICS PERFORMANCE ===
Average Warehouse Processing Time: 4.32 days
Average Wait Time for a Truck:     1.85 days
Average Total Time in Warehouse:   6.18 days
Average Shipping Transit Time:     4.59 days


In [23]:
# 1. Create columns to see what day of the week orders were placed
# (Monday = 0, Tuesday = 1, ..., Saturday = 5, Sunday = 6)
df_master['Order_Day_of_Week'] = df_master['Order Date'].dt.dayofweek
df_master['Order_Day_Name'] = df_master['Order Date'].dt.day_name()

# 2. Check how many orders violate the manager's "2-day processing" claim
# We look at standard orders since Express has different rules
standard_orders = df_master[df_master['Ship Mode'] == 'Standard Class']
delayed_processing_pct = (standard_orders['Days_Order_to_Ready'] > 2).mean() * 100

print("=== CLAIM VERIFICATION ===")
print(f"Percentage of standard orders taking MORE than 2 days to get ready: {delayed_processing_pct:.2f}%")
print("\n" + "="*30 + "\n")

# 3. Analyze the Weekend Bottleneck
# Group by the day the order was placed to see if weekend orders take much longer
weekend_analysis = df_master.groupby('Order_Day_Name')[['Days_Order_to_Ready', 'Total_Warehouse_Days']].mean()

# Reorder the rows logically from Monday to Sunday
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekend_analysis = weekend_analysis.reindex(days_order)

print("=== WEEKEND BOTTLENECK ANALYSIS ===")
print("Average days to process based on the day the customer ordered:")
print(weekend_analysis)


=== CLAIM VERIFICATION ===
Percentage of standard orders taking MORE than 2 days to get ready: 11.23%


=== WEEKEND BOTTLENECK ANALYSIS ===
Average days to process based on the day the customer ordered:
                Days_Order_to_Ready  Total_Warehouse_Days
Order_Day_Name                                           
Monday                     3.805599              6.385251
Tuesday                    4.462462              6.374016
Wednesday                  4.289272              5.973177
Thursday                   4.892489              5.768497
Friday                     3.642686              5.454885
Saturday                   5.481707              6.028761
Sunday                     5.538462              7.885960


In [24]:
# 1. Group the logistics timeline by Ship Mode to compare speeds
ship_mode_comparison = df_master.groupby('Ship Mode')[[
    'Days_Order_to_Ready', 
    'Days_Ready_to_Truck', 
    'Total_Warehouse_Days'
]].mean()

print("=== EXPRESS VS STANDARD PERFORMANCE ===")
print("Average turnaround times (in days) by Shipping Mode:")
print(ship_mode_comparison)
print("\n" + "="*50 + "\n")

# 2. Isolate Express Orders to see how many leave the SAME DAY they are ready
# Same day means Days_Ready_to_Truck should equal 0
express_orders = df_master[df_master['Ship Mode'] == 'First Class'] # or 'Second Class' / 'Express' depending on your values
if len(express_orders) > 0:
    same_day_truck_pct = (express_orders['Days_Ready_to_Truck'] == 0).mean() * 100
    print(f"=== EXPRESS EFFICIENCY ===")
    print(f"Percentage of Express orders loaded onto a truck the SAME DAY they were ready: {same_day_truck_pct:.2f}%")
else:
    print("Check your column values for 'Ship Mode' to ensure the Express label matches your data text.")


=== EXPRESS VS STANDARD PERFORMANCE ===
Average turnaround times (in days) by Shipping Mode:
                Days_Order_to_Ready  Days_Ready_to_Truck  Total_Warehouse_Days
Ship Mode                                                                     
First Class                2.079840             0.379242              2.484885
Second Class               4.283198             1.947154              6.085973
Standard Class             5.078982             2.275457              7.591943


=== EXPRESS EFFICIENCY ===
Percentage of Express orders loaded onto a truck the SAME DAY they were ready: 6.39%


In [25]:
# 1. Filter the dataset to rows where we actually have delivery data 
# Remember, Arrival Scan Date was only collected from the QR code promotion!
delivered_orders = df_master[df_master['Transit_Days'].notnull()]

print("=== GLOBAL CARRIER PERFORMANCE ===")
carrier_avg = delivered_orders['Transit_Days'].mean()
print(f"Overall average shipping transit time: {carrier_avg:.2f} days")
print(f"Percentage of deliveries taking MORE than 3 days: {(delivered_orders['Transit_Days'] > 3).mean() * 100:.2f}%")
print("\n" + "="*50 + "\n")

# 2. Break down transit performance by Region
region_transit = delivered_orders.groupby('Region')['Transit_Days'].agg(['count', 'mean', 'max']).rename(
    columns={'count': 'Scanned Orders', 'mean': 'Avg Transit Days', 'max': 'Longest Delivery (Days)'}
)

print("=== TRANSIT PERFORMANCE BY REGION ===")
print(region_transit.round(2))
print("\n" + "="*50 + "\n")

# 3. Find the top 5 worst States for shipping delays
state_transit = delivered_orders.groupby('State')['Transit_Days'].mean().reset_index()
worst_states = state_transit.sort_values(by='Transit_Days', ascending=False).head(5)

print("=== TOP 5 SLOWEST STATES FOR DELIVERY ===")
print(worst_states.to_string(index=False, formatters={'Transit_Days': '{:,.2f} days'.format}))


=== GLOBAL CARRIER PERFORMANCE ===
Overall average shipping transit time: 4.59 days
Percentage of deliveries taking MORE than 3 days: 80.42%


=== TRANSIT PERFORMANCE BY REGION ===
         Scanned Orders  Avg Transit Days  Longest Delivery (Days)
Region                                                            
Central             627              4.36                      7.0
East                639              4.67                      7.0
South               252              4.62                      7.0
West                693              4.70                      7.0


=== TOP 5 SLOWEST STATES FOR DELIVERY ===
      State Transit_Days
  Louisiana    7.00 days
 New Jersey    6.14 days
Connecticut    6.13 days
Mississippi    5.91 days
   Maryland    5.50 days


In [26]:
import numpy as np

# 1. Isolate the clean, known transit days from your campaign data
known_transit = df_master[df_master['Transit_Days'].notnull()]['Transit_Days'].values

# 2. Find out exactly how many rows are missing delivery data
missing_mask = df_master['Transit_Days'].isnull()
num_missing = missing_mask.sum()

# 3. Monte Carlo Simulation: Randomly sample from known data to fill the gaps
np.random.seed(42) # Set seed so your presentation results are reproducible!
simulated_transit = np.random.choice(known_transit, size=num_missing)

# 4. Inject the simulated data back into a copy of your master dataframe
df_simulated = df_master.copy()
df_simulated.loc[missing_mask, 'Transit_Days'] = simulated_transit

# 5. Calculate the absolute total delivery time from Customer Click to Customer Door
df_simulated['Total_Customer_Wait'] = df_simulated['Total_Warehouse_Days'] + df_simulated['Transit_Days']

# 6. Calculate the 95th Percentile Service Delivery Promise
promise_days = np.percentile(df_simulated['Total_Customer_Wait'].dropna(), 95)

print("=== MONTE CARLO LOGISTICS SIMULATION ===")
print(f"Total rows successfully simulated: {num_missing}")
print(f"Current 95% Service Delivery Promise: {promise_days:.0f} days")
print(f"-> Meaning: 95% of your customers currently receive their muesli within {promise_days:.0f} days.")


=== MONTE CARLO LOGISTICS SIMULATION ===
Total rows successfully simulated: 20960
Current 95% Service Delivery Promise: 15 days
-> Meaning: 95% of your customers currently receive their muesli within 15 days.
